# Diagnóstico de calidad de datos · EnCicla Medellín

Aprendiz: Isabel Cristina Acevedo Guirales · Ficha: 3169901 · Instructor: Carlos Enrique Navia Torres
Actividad: EV06 – Diagnosticar la calidad de datos: preparación, exploración y documentación

## Momento 1. Reconocimiento del proyecto y de los datasets

Proyecto: análisis del uso de las bicicletas públicas EnCicla en Medellín según las condiciones climáticas y las horas pico.

Pregunta analítica: ¿Cómo nos afectan las condiciones climáticas los días de la semana durante las horas pico en la ciudad de Medellín?

Tipo de análisis: descriptivo y diagnóstico.

Variables previstas

| Variable | Qué representa |
| --- | --- |
| `ID_Viaje` | Código único de cada trayecto |
| `Estacion_Origen` / `Estacion_Destino` | Lugar donde empieza y termina el viaje |
| `Hora_Inicio` | Fecha y hora exacta del viaje (clave para extraer las horas pico) |
| `Tipo_Clima` | Estado del tiempo (soleado, nublado, lluvioso) |
| `Temperatura_C` | Temperatura ambiente en grados |
| `Duracion_Viaje_Minutos` | Tiempo que se usó la bicicleta |




| Dataset | Archivo | Fuente | Qué contiene | Relación con la pregunta |
| --- | --- | --- | --- | --- |
| viajes | viajes_encicla_simulado.csv | Datos simulados con scripts/generar_datos_simulados.py | 21.806 viajes (1 ago – 31 oct 2026): ID, estación de origen y destino, hora de inicio y duración | Permite sacar la hora y el día de cada viaje para identificar las horas pico y la demanda por día |
| clima | clima_medellin_simulado.csv | Datos simulados con scripts/generar_datos_simulados.py | 2.160 registros por hora: tipo de clima y temperatura | Aporta las condiciones climáticas para cruzarlas con los viajes por fecha y hora |
Aspectos de calidad que se revisarán antes de usar los datos:

1. Valores faltantes, incluidos los "escondidos" (texto vacío, `N/A`, `-`).
2. Tipos de datos coherentes con lo que representa cada variable (fechas como fecha, medidas como número).
3. Registros duplicados.
4. Valores fuera de un rango razonable (temperatura, duración del viaje).
5. Categorías escritas de varias formas (por ejemplo, `Lluvioso` y `lluvioso `).
6. Cobertura temporal: ¿hay datos de todos los días de la semana y de las horas pico?
7. Columnas comunes que permitan cruzar viajes con clima.

## Momento 2. Preparación del entorno y carga de datos

Los archivos viven en `data/raw/` (no se suben a GitHub, ver `README.md`). Cada dataset se guarda en un DataFrame independiente.


In [1]:
from pathlib import Path
import csv
import warnings

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Raíz del proyecto: funciona si el cuaderno se ejecuta desde notebooks/ o desde la raíz
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = RAIZ / "data" / "raw"

# ── EDITA AQUÍ: nombre de cada archivo dentro de data/raw/ ──────────────────
DATASETS = {
    "viajes": "viajes_encicla_simulado.csv",
    "clima": "clima_medellin_simulado.csv",
}

In [2]:
def leer_archivo(ruta):
    """Carga CSV/Excel/JSON detectando separador y codificación. No modifica los datos."""
    ruta = Path(ruta)
    ext = ruta.suffix.lower()
    if ext in (".xlsx", ".xls"):
        return pd.read_excel(ruta)
    if ext == ".json":
        return pd.read_json(ruta)
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            with open(ruta, encoding=enc, newline="") as f:
                muestra = f.read(20000)
            try:
                sep = csv.Sniffer().sniff(muestra, delimiters=",;\t|").delimiter
            except csv.Error:
                sep = ","
            df = pd.read_csv(ruta, sep=sep, encoding=enc)
            print(f"   (separador {sep!r}, codificación {enc})")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No se pudo leer {ruta.name}")


dfs = {}
for nombre, archivo in DATASETS.items():
    ruta = RAW / archivo
    if not ruta.exists():
        print(f"✗ {nombre}: no existe {ruta}")
        print("   Archivos en data/raw:", sorted(p.name for p in RAW.glob("*")))
        continue
    dfs[nombre] = leer_archivo(ruta)
    print(f"✓ {nombre}: {dfs[nombre].shape[0]:,} filas × {dfs[nombre].shape[1]} columnas")

assert dfs, "No se cargó ningún dataset: revisa los nombres en DATASETS"

   (separador ',', codificación utf-8)
✓ viajes: 21,806 filas × 5 columnas
   (separador ',', codificación utf-8)
✓ clima: 2,160 filas × 3 columnas


**Verificación de la carga:** si algún dataset muestra **1 columna**, el separador no se detectó bien; si muestra símbolos raros en las tildes, revisa la codificación.

## Momento 3. Inspección de la estructura de los datasets

Se aplican a cada DataFrame los comandos pedidos: `head()`, `tail()`, `shape`, `columns`, `dtypes`, `info()`, `describe()` y `describe(include='object')`.

### 3.1 `head()` · primeros registros

In [3]:
for nombre, df in dfs.items():
    print(f"── {nombre}")
    display(df.head())

── viajes


,ID_Viaje,Estacion_Origen,Estacion_Destino,Hora_Inicio,Duracion_Viaje_Minutos
0,V000001,Hospital Infantil Concejo de Medellín,Estadio,2026-08-01 04:17:00,8
1,V000002,Hospital Infantil Concejo de Medellín,Hospital Infantil Concejo de Medellín,2026-08-01 04:39:00,11
2,V000003,Universidad,Universidad,2026-08-01 04:33:00,5
3,V000004,Universidad,Floresta,2026-08-01 05:15:00,7
4,V000005,Universidad,Hospital Infantil Concejo de Medellín,2026-08-01 05:54:00,8


── clima


,Fecha_Hora,Tipo_Clima,Temperatura_C
0,2026-08-01 00:00:00,Nublado,16.0
1,2026-08-01 01:00:00,Soleado,16.8
2,2026-08-01 02:00:00,Soleado,14.1
3,2026-08-01 03:00:00,Nublado,15.4
4,2026-08-01 04:00:00,Nublado,15.2


### 3.2 `tail()` · últimos registros

In [4]:
for nombre, df in dfs.items():
    print(f"── {nombre}")
    display(df.tail())

── viajes


,ID_Viaje,Estacion_Origen,Estacion_Destino,Hora_Inicio,Duracion_Viaje_Minutos
21801,V015822,Suramericana,Suramericana,2026-10-07 07:18:00,41
21802,V013492,Hospital Infantil Concejo de Medellín,Estadio,2026-09-27 10:59:00,24
21803,V003596,Suramericana,Estadio,2026-08-16 10:40:00,15
21804,V016819,Suramericana,Suramericana,2026-10-11 13:00:00,16
21805,V008152,Suramericana,Estadio,2026-09-04 07:15:00,15


── clima


,Fecha_Hora,Tipo_Clima,Temperatura_C
2155,2026-10-31 19:00:00,Lluvioso,21.1
2156,2026-10-31 20:00:00,Soleado,19.5
2157,2026-10-31 21:00:00,Soleado,20.9
2158,2026-10-31 22:00:00,Soleado,21.2
2159,2026-10-31 23:00:00,Nublado,18.2


### 3.3 `shape` · filas y columnas

In [5]:
for nombre, df in dfs.items():
    print(f"{nombre}: {df.shape[0]:,} filas × {df.shape[1]} columnas")

viajes: 21,806 filas × 5 columnas
clima: 2,160 filas × 3 columnas


### 3.4 `columns` · nombres de las variables

In [6]:
for nombre, df in dfs.items():
    print(f"── {nombre}")
    print(list(df.columns))

── viajes
['ID_Viaje', 'Estacion_Origen', 'Estacion_Destino', 'Hora_Inicio', 'Duracion_Viaje_Minutos']
── clima
['Fecha_Hora', 'Tipo_Clima', 'Temperatura_C']


### 3.5 `dtypes` · tipos de datos

In [7]:
for nombre, df in dfs.items():
    print(f"── {nombre}")
    display(df.dtypes.astype(str).rename("tipo").to_frame())

── viajes


,tipo
ID_Viaje,str
Estacion_Origen,str
Estacion_Destino,str
Hora_Inicio,str
Duracion_Viaje_Minutos,int64


── clima


,tipo
Fecha_Hora,str
Tipo_Clima,str
Temperatura_C,float64


### 3.6 `info()` · estructura y valores no nulos

In [8]:
for nombre, df in dfs.items():
    print(f"── {nombre}")
    df.info(show_counts=True)
    print()

── viajes
<class 'pandas.DataFrame'>
RangeIndex: 21806 entries, 0 to 21805
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   ID_Viaje                21806 non-null  str  
 1   Estacion_Origen         21746 non-null  str  
 2   Estacion_Destino        21806 non-null  str  
 3   Hora_Inicio             21806 non-null  str  
 4   Duracion_Viaje_Minutos  21806 non-null  int64
dtypes: int64(1), str(4)
memory usage: 851.9 KB

── clima
<class 'pandas.DataFrame'>
RangeIndex: 2160 entries, 0 to 2159
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Fecha_Hora     2160 non-null   str    
 1   Tipo_Clima     2098 non-null   str    
 2   Temperatura_C  2011 non-null   float64
dtypes: float64(1), str(2)
memory usage: 50.8 KB



### 3.7 `describe()` · estadísticas de las variables numéricas

In [9]:
for nombre, df in dfs.items():
    print(f"── {nombre}")
    if df.select_dtypes(include="number").shape[1]:
        display(df.describe())
    else:
        print("Sin columnas numéricas")

── viajes


,Duracion_Viaje_Minutos
count,21806.000000
mean,14.492112
std,26.806744
min,-3.000000
25%,9.000000
50%,12.000000
75%,17.000000
max,1439.000000


── clima


,Temperatura_C
count,2011.000000
mean,20.864993
std,8.845023
min,10.500000
25%,16.600000
50%,20.300000
75%,23.500000
max,99.900000


### 3.8 `describe(include='object')` · variables de texto

In [10]:
# Nota: en pandas 3 el texto puede venir con tipo 'str' en lugar de 'object'; se incluyen ambos.
for nombre, df in dfs.items():
    print(f"── {nombre}")
    texto = df.select_dtypes(include=["object", "string"])
    if texto.shape[1]:
        display(texto.describe())
    else:
        print("Sin columnas de texto")

── viajes


,ID_Viaje,Estacion_Origen,Estacion_Destino,Hora_Inicio
count,21806,21746,21806,21806
unique,21766,6,6,18991
top,V001127,Universidad,Universidad,sin dato
freq,2,5589,5498,15


── clima


,Fecha_Hora,Tipo_Clima
count,2160,2098
unique,2160,7
top,2026-08-01 00:00:00,Soleado
freq,1,1109


### 3.9 Tabla de diagnóstico estructural

In [11]:
def resumen_tipos(df):
    return ", ".join(f"{t}: {n}" for t, n in df.dtypes.astype(str).value_counts().items())


diagnostico = pd.DataFrame(
    [
        {
            "Dataset": nombre,
            "Filas": df.shape[0],
            "Columnas": df.shape[1],
            "Tipos de datos": resumen_tipos(df),
            "Observaciones": f"{int(df.isna().sum().sum()):,} valores nulos; "
                             f"{int(df.duplicated().sum()):,} filas duplicadas",
        }
        for nombre, df in dfs.items()
    ]
)
display(diagnostico)

,Dataset,Filas,Columnas,Tipos de datos,Observaciones
0,viajes,21806,5,"str: 4, int64: 1",60 valores nulos; 40 filas duplicadas
1,clima,2160,3,"str: 2, float64: 1",211 valores nulos; 0 filas duplicadas


### Interpretación de la inspección estructural

El dataset de viajes tiene 21.806 filas y 5 columnas; cuatro son texto y solo la duración es numérica. Cubre del 1 de agosto al 31 de octubre de 2026. El dataset de clima tiene 2.160 filas y 3 columnas, con un registro por hora. Los dos archivos no comparten columnas y sus fechas están guardadas como texto, así que habrá que unirlos por fecha y hora.principales de cada dataset (tamaño, qué variables tiene, qué tipos predominan, qué llama la atención en `head()`/`tail()`)._

## Momento 4. Diagnóstico inicial de calidad

Solo se **diagnostica y documenta**: ningún DataFrame se modifica. Los hallazgos objetivos se van registrando en una lista y al final se ordenan en la tabla pedida (`Dataset · Problema · Evidencia · Posible impacto · Acción propuesta`).

**Primero ajusta la configuración** con los nombres reales de tus columnas (los viste en el punto 3.4) y los umbrales que consideres razonables. Si una variable no existe en un dataset, esa revisión simplemente se omite para él.

In [12]:
# ── EDITA AQUÍ: nombre real de cada variable en tus archivos (None si no existe) ──
COLUMNAS = {
    "id": "ID_Viaje",
    "origen": "Estacion_Origen",
    "destino": "Estacion_Destino",
    "inicio": "Hora_Inicio",
    "clima": "Tipo_Clima",
    "temp": "Temperatura_C",
    "duracion": "Duracion_Viaje_Minutos",
}

# ── Umbrales de revisión (criterio propio: justifícalos en el informe) ──────────
TEMP_MIN, TEMP_MAX = 5, 40          # °C plausibles para Medellín
DUR_MIN, DUR_MAX = 1, 180           # minutos de un viaje razonable
HORAS_PICO = [6, 7, 8, 17, 18, 19]  # horas que consideras "pico"

VALORES_SOSPECHOSOS = ["", "na", "n/a", "nan", "null", "none", "-", "--", "?", "sin dato", "s/d"]
DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

hallazgos = []


def registrar(dataset, problema, evidencia, impacto, accion):
    hallazgos.append(
        {
            "Dataset": dataset,
            "Problema identificado": problema,
            "Evidencia encontrada": evidencia,
            "Posible impacto": impacto,
            "Acción propuesta": accion,
        }
    )


def col(df, rol):
    """Nombre real de la columna para un rol, o None si no está en este dataset."""
    nombre = COLUMNAS.get(rol)
    return nombre if nombre in df.columns else None


def columnas_texto(df):
    return list(df.select_dtypes(include=["object", "string"]).columns)


def a_fecha(serie):
    if pd.api.types.is_datetime64_any_dtype(serie):
        return serie
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return pd.to_datetime(serie, errors="coerce")


def a_numero(serie):
    return serie if pd.api.types.is_numeric_dtype(serie) else pd.to_numeric(serie, errors="coerce")

### 4.1 Valores faltantes

In [13]:
for nombre, df in dfs.items():
    print(f"── {nombre}")
    nulos = df.isna().sum()
    tabla = pd.DataFrame({"nulos": nulos, "% nulos": (nulos / len(df) * 100).round(2)})
    tabla = tabla[tabla["nulos"] > 0].sort_values("nulos", ascending=False)
    if tabla.empty:
        print("Sin valores nulos (NaN).")
    else:
        display(tabla)
        evidencia = "; ".join(f"{c} ({int(r['nulos']):,}, {r['% nulos']}%)" for c, r in tabla.iterrows())
        registrar(
            nombre, "Valores faltantes (NaN)", evidencia,
            "Reduce los registros utilizables y puede sesgar promedios y conteos por clima u hora.",
            "Decidir por columna si se eliminan filas, se imputan o se dejan, según su importancia para la pregunta.",
        )

    # Nulos "escondidos": texto vacío, N/A, '-', etc.
    ocultos = {}
    for c in columnas_texto(df):
        limpio = df[c].dropna().astype(str).str.strip().str.lower()
        n = int(limpio.isin(VALORES_SOSPECHOSOS).sum())
        if n:
            ocultos[c] = n
    if ocultos:
        print("Posibles nulos escondidos (texto vacío, 'N/A', '-'...):", ocultos)
        registrar(
            nombre, "Nulos escondidos en columnas de texto", str(ocultos),
            "Pandas no los cuenta como nulos y pueden pasar como categorías válidas.",
            "Convertirlos a NaN con replace() antes de analizar.",
        )

── viajes


,nulos,% nulos
Estacion_Origen,60,0.28


Posibles nulos escondidos (texto vacío, 'N/A', '-'...): {'Hora_Inicio': 15}
── clima


,nulos,% nulos
Temperatura_C,149,6.90
Tipo_Clima,62,2.87


### 4.2 Tipos de datos coherentes con lo que representan

In [14]:
TIPO_ESPERADO = {"inicio": "fecha", "temp": "número", "duracion": "número", "id": "id/texto", "clima": "categoría",
                 "origen": "categoría", "destino": "categoría"}
filas = []
for nombre, df in dfs.items():
    for rol, esperado in TIPO_ESPERADO.items():
        c = col(df, rol)
        if c is None:
            continue
        actual = str(df[c].dtype)
        ok = True
        if esperado == "fecha":
            ok = pd.api.types.is_datetime64_any_dtype(df[c])
            if not ok:
                parseadas = a_fecha(df[c])
                fallidas = int((parseadas.isna() & df[c].notna()).sum())
                registrar(
                    nombre, f"'{c}' no es de tipo fecha (tipo {actual})",
                    f"pd.to_datetime la interpreta; {fallidas:,} valores no se pudieron convertir",
                    "No se pueden extraer hora y día de la semana, base de la pregunta analítica.",
                    f"Convertir '{c}' con pd.to_datetime y crear columnas de hora y día de la semana.",
                )
        elif esperado == "número":
            ok = pd.api.types.is_numeric_dtype(df[c])
            if not ok:
                fallidas = int((a_numero(df[c]).isna() & df[c].notna()).sum())
                registrar(
                    nombre, f"'{c}' debería ser numérica pero es {actual}",
                    f"{fallidas:,} valores no se pueden convertir a número",
                    "No permite calcular promedios ni estadísticas.",
                    f"Limpiar el texto y convertir '{c}' con pd.to_numeric.",
                )
        filas.append({"Dataset": nombre, "Columna": c, "Rol": rol, "Tipo actual": actual,
                      "Tipo esperado": esperado, "¿Coherente?": "sí" if ok else "NO"})
display(pd.DataFrame(filas) if filas else "Ninguna de las variables de COLUMNAS existe en los datasets: revisa la configuración.")

,Dataset,Columna,Rol,Tipo actual,Tipo esperado,¿Coherente?
0,viajes,Hora_Inicio,inicio,str,fecha,NO
1,viajes,Duracion_Viaje_Minutos,duracion,int64,número,sí
2,viajes,ID_Viaje,id,str,id/texto,sí
3,viajes,Estacion_Origen,origen,str,categoría,sí
4,viajes,Estacion_Destino,destino,str,categoría,sí
5,clima,Temperatura_C,temp,float64,número,sí
6,clima,Tipo_Clima,clima,str,categoría,sí


### 4.3 Registros duplicados

In [15]:
for nombre, df in dfs.items():
    dup_filas = int(df.duplicated().sum())
    print(f"── {nombre}: {dup_filas:,} filas completamente duplicadas")
    if dup_filas:
        registrar(
            nombre, "Filas completamente duplicadas", f"{dup_filas:,} de {len(df):,} filas",
            "Infla el conteo de viajes y distorsiona la demanda por día y hora.",
            "Eliminar duplicados con drop_duplicates() tras confirmar que son errores.",
        )
    c = col(df, "id")
    if c:
        dup_id = int(df[c].duplicated().sum())
        print(f"   '{c}' repetido en {dup_id:,} filas")
        if dup_id:
            registrar(
                nombre, f"Identificadores repetidos en '{c}'", f"{dup_id:,} filas con id ya visto",
                "Un mismo viaje podría estar contado más de una vez.",
                "Revisar si son el mismo viaje y conservar un solo registro.",
            )

── viajes: 40 filas completamente duplicadas
   'ID_Viaje' repetido en 40 filas
── clima: 0 filas completamente duplicadas


### 4.4 Valores fuera de rango (temperatura y duración)

In [16]:
for nombre, df in dfs.items():
    c = col(df, "temp")
    if c:
        t = a_numero(df[c])
        fuera = int(((t < TEMP_MIN) | (t > TEMP_MAX)).sum())
        print(f"── {nombre} · {c}: min={t.min()}, max={t.max()}, fuera de [{TEMP_MIN}, {TEMP_MAX}] °C: {fuera:,}")
        if fuera:
            registrar(
                nombre, f"Temperaturas fuera de rango en '{c}'", f"{fuera:,} valores fuera de {TEMP_MIN}–{TEMP_MAX} °C",
                "Pueden ser errores de sensor o de unidad y alteran el análisis por clima.",
                "Revisar los casos y marcar o descartar los imposibles.",
            )
    c = col(df, "duracion")
    if c:
        d = a_numero(df[c])
        fuera = int(((d < DUR_MIN) | (d > DUR_MAX)).sum())
        print(f"── {nombre} · {c}: min={d.min()}, max={d.max()}, fuera de [{DUR_MIN}, {DUR_MAX}] min: {fuera:,}")
        if fuera:
            registrar(
                nombre, f"Duraciones fuera de rango en '{c}'", f"{fuera:,} viajes fuera de {DUR_MIN}–{DUR_MAX} min",
                "Viajes de 0 min o de muchas horas suelen ser errores o bicicletas no devueltas; sesgan el promedio.",
                "Definir un criterio de exclusión y documentarlo.",
            )

── viajes · Duracion_Viaje_Minutos: min=-3, max=1439, fuera de [1, 180] min: 45
── clima · Temperatura_C: min=10.5, max=99.9, fuera de [5, 40] °C: 20


### 4.5 Categorías inconsistentes (clima y estaciones)

In [17]:
for nombre, df in dfs.items():
    c = col(df, "clima")
    if c:
        crudo = df[c].dropna().astype(str)
        normalizado = crudo.str.strip().str.lower()
        print(f"── {nombre} · {c}: {crudo.nunique()} valores distintos, {normalizado.nunique()} tras limpiar mayúsculas/espacios")
        display(df[c].value_counts(dropna=False).head(15).rename("registros").to_frame())
        if crudo.nunique() != normalizado.nunique():
            registrar(
                nombre, f"Categorías de '{c}' escritas de varias formas",
                f"{crudo.nunique()} valores distintos pero {normalizado.nunique()} reales",
                "Separa en grupos distintos lo que es la misma condición climática.",
                "Normalizar con str.strip().str.lower() y unificar nombres.",
            )
    o, d = col(df, "origen"), col(df, "destino")
    if o and d:
        igual = int((df[o] == df[d]).sum())
        print(f"── {nombre}: {df[o].nunique()} estaciones de origen, {df[d].nunique()} de destino; "
              f"origen = destino en {igual:,} viajes (puede ser normal, revisar)")

── viajes: 6 estaciones de origen, 6 de destino; origen = destino en 4,033 viajes (puede ser normal, revisar)
── clima · Tipo_Clima: 7 valores distintos, 3 tras limpiar mayúsculas/espacios


,registros
Tipo_Clima,
Soleado,1109
Nublado,575
Lluvioso,326
NaN,62
LLUVIOSO,25
lluvioso,24
NUBLADO,21
soleado,18


### 4.6 Cobertura temporal: días de la semana y horas pico

In [18]:
for nombre, df in dfs.items():
    c = col(df, "inicio")
    if not c:
        continue
    f = a_fecha(df[c]).dropna()
    if f.empty:
        print(f"── {nombre}: no se pudo interpretar ninguna fecha en '{c}'")
        continue
    dias_presentes = sorted(f.dt.day_of_week.unique())
    horas = sorted(f.dt.hour.unique())
    en_pico = int(f.dt.hour.isin(HORAS_PICO).sum())
    print(f"── {nombre} · {c}")
    print(f"   Rango: {f.min()} → {f.max()} ({f.dt.normalize().nunique():,} días distintos)")
    print(f"   Días de la semana presentes: {[DIAS[i] for i in dias_presentes]}")
    print(f"   Horas distintas con datos: {len(horas)} de 24 · registros en horas pico {HORAS_PICO}: {en_pico:,} ({en_pico/len(f):.1%})")
    if len(dias_presentes) < 7 or any(h not in horas for h in HORAS_PICO):
        registrar(
            nombre, "Cobertura incompleta de días u horas pico",
            f"{len(dias_presentes)} de 7 días; horas pico sin datos: {[h for h in HORAS_PICO if h not in horas]}",
            "No se podría comparar el comportamiento entre días de la semana u horas pico.",
            "Buscar datos complementarios o acotar la pregunta a lo que sí está cubierto.",
        )

── viajes · Hora_Inicio
   Rango: 2026-08-01 04:17:00 → 2026-10-31 22:53:00 (92 días distintos)
   Días de la semana presentes: ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
   Horas distintas con datos: 24 de 24 · registros en horas pico [6, 7, 8, 17, 18, 19]: 11,709 (53.7%)


### 4.7 Estructura general: columnas sobrantes, constantes y columnas comunes

In [19]:
for nombre, df in dfs.items():
    sobrantes = [c for c in df.columns if str(c).startswith("Unnamed")]
    constantes = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    print(f"── {nombre}: columnas 'Unnamed' {sobrantes or 'ninguna'} · columnas constantes {constantes or 'ninguna'}")
    if sobrantes:
        registrar(nombre, "Columna índice sobrante", str(sobrantes),
                  "Ruido sin información.", "Eliminarla al preparar los datos.")
    if constantes:
        registrar(nombre, "Columnas con un único valor", str(constantes),
                  "No aportan información al análisis.", "Descartarlas si no son necesarias.")

nombres = list(dfs)
for i in range(len(nombres)):
    for j in range(i + 1, len(nombres)):
        comunes = sorted(set(dfs[nombres[i]].columns) & set(dfs[nombres[j]].columns))
        print(f"Columnas comunes entre '{nombres[i]}' y '{nombres[j]}':", comunes or "ninguna (hay que construir la llave, p. ej. fecha y hora)")

── viajes: columnas 'Unnamed' ninguna · columnas constantes ninguna
── clima: columnas 'Unnamed' ninguna · columnas constantes ninguna
Columnas comunes entre 'viajes' y 'clima': ninguna (hay que construir la llave, p. ej. fecha y hora)


### 4.8 Tabla de hallazgos

In [20]:
f = pd.to_datetime(dfs["clima"]["Fecha_Hora"])
esperado = pd.date_range(f.min(), f.max(), freq="h")
faltan = esperado.difference(f)
print(f"Esperadas {len(esperado):,} horas, hay {len(f):,}, faltan {len(faltan)}")
print("Primera y última hora faltante:", faltan.min(), faltan.max())

registrar("clima", "Horas sin registro", f"{len(faltan)} horas faltantes entre {faltan.min()} y {faltan.max()}",
"No se podrá asignar clima a los viajes de esas horas.", "Buscar otra fuente para completar o excluir esos viajes.")
registrar("clima", "'Fecha_Hora' guardada como texto (str)", "dtypes muestra str",
"No se puede cruzar con los viajes por fecha y hora.", "Convertir con pd.to_datetime y redondear Hora_Inicio a la hora.")

Esperadas 2,208 horas, hay 2,160, faltan 48
Primera y última hora faltante: 2026-09-11 16:00:00 2026-09-13 15:00:00


In [21]:
tabla_hallazgos = pd.DataFrame(hallazgos)
if tabla_hallazgos.empty:
    print("No se registraron problemas automáticos: confirma revisando los resultados anteriores.")
else:
    display(tabla_hallazgos)
    (RAIZ / "reports").mkdir(exist_ok=True)
    tabla_hallazgos.to_csv(RAIZ / "reports" / "hallazgos_calidad.csv", index=False, encoding="utf-8-sig")
    print(f"\n{len(tabla_hallazgos)} hallazgos guardados en reports/hallazgos_calidad.csv")

,Dataset,Problema identificado,Evidencia encontrada,Posible impacto,Acción propuesta
0,viajes,Valores faltantes (NaN),"Estacion_Origen (60, 0.28%)",Reduce los registros utilizables y puede sesga...,"Decidir por columna si se eliminan filas, se i..."
1,viajes,Nulos escondidos en columnas de texto,{'Hora_Inicio': 15},Pandas no los cuenta como nulos y pueden pasar...,Convertirlos a NaN con replace() antes de anal...
2,clima,Valores faltantes (NaN),"Temperatura_C (149, 6.9%); Tipo_Clima (62, 2.87%)",Reduce los registros utilizables y puede sesga...,"Decidir por columna si se eliminan filas, se i..."
3,viajes,'Hora_Inicio' no es de tipo fecha (tipo str),pd.to_datetime la interpreta; 15 valores no se...,"No se pueden extraer hora y día de la semana, ...",Convertir 'Hora_Inicio' con pd.to_datetime y c...
4,viajes,Filas completamente duplicadas,"40 de 21,806 filas",Infla el conteo de viajes y distorsiona la dem...,Eliminar duplicados con drop_duplicates() tras...
5,viajes,Identificadores repetidos en 'ID_Viaje',40 filas con id ya visto,Un mismo viaje podría estar contado más de una...,Revisar si son el mismo viaje y conservar un s...
6,viajes,Duraciones fuera de rango en 'Duracion_Viaje_M...,45 viajes fuera de 1–180 min,Viajes de 0 min o de muchas horas suelen ser e...,Definir un criterio de exclusión y documentarlo.
7,clima,Temperaturas fuera de rango en 'Temperatura_C',20 valores fuera de 5–40 °C,Pueden ser errores de sensor o de unidad y alt...,Revisar los casos y marcar o descartar los imp...
8,clima,Categorías de 'Tipo_Clima' escritas de varias ...,7 valores distintos pero 3 reales,Separa en grupos distintos lo que es la misma ...,Normalizar con str.strip().str.lower() y unifi...
9,clima,Horas sin registro,48 horas faltantes entre 2026-09-11 16:00:00 y...,No se podrá asignar clima a los viajes de esas...,Buscar otra fuente para completar o excluir es...



11 hallazgos guardados en reports/hallazgos_calidad.csv


### Interpretación del diagnóstico de calidad

La tabla anterior es un **borrador automático**: revísala. Agrega los problemas que veas a simple vista y que el código no detecta, y ajusta el impacto y la acción a tu pregunta analítica.

Los problemas más graves para la pregunta son: Hora_Inicio es texto y tiene 15 valores "sin dato", lo que impide sacar la hora y el día; el clima tiene 149 temperaturas vacías, 20 valores imposibles (99,9) y el tipo de clima escrito de 7 formas, lo que separaría grupos que son iguales; y faltan 48 horas de clima. También hay 40 viajes duplicados y 45 duraciones fuera de rango. Es menos grave que 60 estaciones de origen estén vacías (0,28 %). Como punto a favor, hay datos de los 7 días y de las 24 horas.

**Acciones de limpieza y preparación propuestas** (no es obligatorio ejecutarlas en esta actividad):

1) Convertir Hora_Inicio y Fecha_Hora a fecha y crear columnas de hora y día de la semana. 2) Eliminar los 40 duplicados. 3) Normalizar Tipo_Clima con minúsculas y sin espacios. 4) Reemplazar 99,9 y "sin dato" por valores vacíos y decidir cómo tratar los vacíos. 5) Revisar las duraciones fuera de 1–180 minutos. 6) Redondear los viajes a la hora y unirlos con el clima. 7) Buscar otra fuente para las 48 horas faltantes.

## Conclusiones sobre las condiciones actuales de los datos

Los datos permiten continuar con el proyecto, pero solo después de una limpieza previa. Cubren los días y las horas pico que pide la pregunta, aunque los viajes y el clima están separados y el clima tiene huecos. Además, son datos simulados, así que los resultados sirven para practicar y no para sacar conclusiones sobre EnCicla.